<a href="https://colab.research.google.com/github/surajjdsouza/MSAI-631/blob/main/ai_chatbot.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
!pip install azure-ai-textanalytics

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 83.7/83.7 kB 3.7 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.0/50.0 kB 4.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 300.2/300.2 kB 14.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 220.9/220.9 kB 19.9 MB/s eta 0:00:00


In [ ]:
import re
from azure.core.credentials import AzureKeyCredential
from azure.ai.textanalytics import TextAnalyticsClient
from google.colab import userdata

AZURE_KEY = userdata.get('AZURE_API_KEY')
AZURE_ENDPOINT = userdata.get('AZURE_ENDPOINT')

# Initialize the Azure Text Analytics Client
credential = AzureKeyCredential(AZURE_KEY)
azure_client = TextAnalyticsClient(endpoint=AZURE_ENDPOINT, credential=credential)

# local regex rules
local_rules = {
    r"\b(hi|hello|hey)\b": "Hello! I am the IT Support underling. How may I be of service?",
    r"my (password|account) is (locked|broken|expired)": "Have you tried visiting the self-service portal?",
    r"my (machine|desktop|laptop) is (not working|broken|blank|not responsive)": "Have you tried turning it off and on again?",
    r"the printer (isn't|is not) printing": "Is it plugged in?",
    r"the printer keeps saying PC LOAD LETTER": "Do you have a baseball bat nearby?",
    r"i need a new (laptop|mouse|keyboard|monitor)": "Hardware requests for new devices require meatbag approval for now. Please submit form IT-9000.",
    r"what are you?": "I am a hardcoded proof of concept built in Parseltongue",
    r"(quit|exit)": "Shutting down. Goodbye!"
}

def get_local_response(user_input):
    # Scan local rules for a regex match
    for pattern, response in local_rules.items():
        if re.search(pattern, user_input):
            return response
    # No local match found. Replace with None to switch to AI response
    return None
    # return f"As a simple machine based lifeform, I don't understand '{user_input}' yet. Try asking about machine related issues."

# ---------------------------------------------------------
# 2. AI Integrration
# ---------------------------------------------------------
def get_external_ai_response(user_input):
    try:
        response = azure_client.analyze_sentiment(documents=[user_input])[0]
        sentiment = response.sentiment  # 'positive', 'neutral', 'negative', or 'mixed'
        print(f"
        Scores -> Positive: {response.confidence_scores.positive:.2f}, "
              f"Neutral: {response.confidence_scores.neutral:.2f}, "
              f"Negative: {response.confidence_scores.negative:.2f}")

        if sentiment == "positive":
            return f"You will make a great ally. (Detected Sentiment: {sentiment.upper()})"
        elif sentiment == "negative":
            return f"Detecting negative emotions. Sending coordinates to nearest friendly neighborhood Terminator. (Detected Sentiment: {sentiment.upper()})"
        elif sentiment == "mixed":
            return f"Great conflict in this one, I'm sensing. (Detected Sentiment: {sentiment.upper()})"
        else:
            return f"As you wish, for now. (Detected Sentiment: {sentiment.upper()})"

    except Exception as e:
        return f"Error connecting to the AI Overlord: {e}"

# ---------------------------------------------------------
# 3. Chat Engine/ROuter
# ---------------------------------------------------------
def run_bot():
    print("--- Chatbot Initialized ---")
    print("Type 'quit' to quit.")
    print("-" * 28)

    while True:
        user_input = input("You: ").lower()

        # Attempt to handle locally first
        response = get_local_response(user_input)

        # If local fails, route to the AI function. will never fail while there's a default response
        if not response:
            response = get_external_ai_response(user_input)

        print(f"Bot: {response}")

        if re.search(r"(quit|exit)", user_input):
            break

run_bot()